# Clinical Cluster Characterisation
**T2D Consensus Clusters — NHANES 1999–2018**

**Purpose.** Describe the three consensus clusters in clinical terms, map them onto the
Ahlqvist et al. (2018) diabetes subtypes, test whether they differ on variables that were *not*
used to derive them, and quantify cause-specific mortality by subtype. This is where
statistical clusters become clinical phenotypes.

**Inputs.** `consensus_k3_clusters.csv` from the clustering notebook, and — for Section 16 only
— `diabetes_with_mortality.csv`, the cohort joined to the NHANES public-use linked mortality
file. The second file is **not produced by this notebook**; it must exist before Section 16 runs.

**Outputs.** `t2d_ahlqvist_mapped.csv` (cohort plus subtype labels and all derived clinical
categories), `table1_clinical_summary.csv`, `table1_baseline_characteristics.csv`,
`table2_mortality_rates.csv`, and figures in `figures_clinical/`.

**How to run.** Top to bottom. Sections depend on objects created earlier (`subtype_map`,
`CLUSTER_NAMES`, `df_mort`), so cells are not independently runnable. Requires `numpy`,
`pandas`, `scipy`, `matplotlib` and `seaborn`.

---

## What this notebook is doing, and why it matters

The clusters were derived from five features: Age, BMI, HbA1c, HOMA-IR and HOMA-β. Any test
showing that the clusters differ on those five is close to circular — it restates the
clustering rather than validating it.

The informative comparisons are therefore the variables that played **no part** in deriving
the clusters: kidney function (eGFR), blood pressure, lipids, diabetes duration, and above all
mortality. Differences on those are genuine external validation. When reading the results,
keep the two categories separate.

**Subtype assignment (Section 9)** uses a greedy, priority-ordered rule on the cluster means:

1. **SIDD** — the cluster with the lowest HOMA-β (beta-cell failure is the least ambiguous
   single-feature signature)
2. **SOIRD** — of those remaining, the highest HOMA-IR
3. **MOD / MARD** — whatever is left: MOD if mean BMI ≥ 30, otherwise MARD

Each subtype is claimed on **one** feature, then removed from the pool. This is transparent
and reproducible, but it does not verify that the assigned cluster matches the *rest* of the
expected profile — SIDD should also be younger with higher HbA1c, for instance. Check the
Z-score heatmap and radar in Section 9 to confirm the full profile is consistent before
adopting the labels.

---

## Known issues to review before use

Spotted while documenting and **left unchanged**, since the analysis logic was preserved as-is:

2. **The Kaplan–Meier curves have no confidence bands and no at-risk table.** The hand-rolled
   estimator returns point estimates only. Most journals expect both for a survival figure,
   and without CIs a reader cannot judge whether visibly separated curves are distinguishable.
   Note also that the product-limit update is correct for ties, but when a censoring time
   exactly equals an event time the sort order between them is arbitrary, which introduces a
   very small bias.
4. **Axis labels are inconsistent across the figure set.** `CLUSTER_NAMES` is only populated in
   Section 9, but Sections 4–8 plot before that. Figures 2, 3, 5 and 6 therefore carry plain
   cluster numbers while Figures 9–14 carry subtype names. Re-running Sections 4–8 after
   Section 9 (or moving the subtype mapping earlier) would make the whole set consistent.

6. **Subtype mapping uses arithmetic means of skewed variables.** `profile` is built from
   `.mean()` on raw HOMA-IR and HOMA-β, which are right-skewed, whereas the clustering itself
   used log-transformed values. The mapping is unlikely to change, but the means shown are not
   the centroids the algorithm actually saw.
7. **The Z-score heatmap has a compressed range.** Standardising across only K cluster means
   bounds |z| at √(K−1) ≈ 1.41 for K=3. The colours show relative position, not participant-level
   z-scores — worth a figure-legend note so readers do not over-interpret the magnitudes.

> Cell outputs have been cleared so the notebook is light and diffs cleanly. Re-run top to
> bottom to regenerate them.

---

## Notebook contents

| Section | Content |
|---------|---------|
| 1 | Imports, helpers & statistical functions |
| 2 | Load data |
| 3 | Diabetes status distribution |
| 4 | HOMA-IR & HOMA-B classification |
| 5 | BMI & Age analysis |
| 6 | Sex distribution |
| 7 | eGFR / kidney health |
| 8 | HbA1c classification |
| 9 | Clinical profiles & Ahlqvist subtype mapping |
| 10 | Statistical tests — Kruskal-Wallis & Dunn |
| 11 | Table 1 — clinical summary |
| 11b | Comprehensive baseline characteristics |
| 12 | Lipid profile |
| 13 | Blood pressure |
| 14 | Ethnicity & lifestyle |
| 15 | Diabetes duration |
| 16 | Mortality analysis by subtype |


## 1 · Imports & Publication Style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
# CAUTION: suppresses ALL warnings, including the seaborn `palette`-without-`hue`
# deprecation triggered by violin_box() below, and any RuntimeWarning from the
# statistical tests. Comment out when upgrading libraries or debugging.
warnings.filterwarnings("ignore")

from scipy.stats import kruskal, chi2_contingency
from itertools import combinations

# ── Output folder ─────────────────────────────────────────────────────────────
FIGURE_DIR = "figures_clinical"
os.makedirs(FIGURE_DIR, exist_ok=True)   # exist_ok so re-runs never error

# ── Publication rcParams ──────────────────────────────────────────────────────
# Set once so every figure is visually consistent and export-ready, rather than
# repeating styling arguments in each plotting cell.
PUB_RC = {
    "font.family":         "sans-serif",
    "font.sans-serif":     ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":           10,
    "axes.titlesize":      11,
    "axes.labelsize":      10,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "legend.fontsize":     9,
    "legend.title_fontsize": 9,
    "lines.linewidth":     1.5,
    "patch.linewidth":     0.8,
    "axes.spines.top":     False,   # drop non-data ink
    "axes.spines.right":   False,
    "axes.linewidth":      0.8,
    "axes.grid":           False,
    "figure.dpi":          150,     # screen preview
    "savefig.dpi":         300,     # journal minimum for raster output
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.05,
    "pdf.fonttype":        42,      # embed fonts — required by most journals
    "ps.fonttype":         42,
    "svg.fonttype":        "none",  # keep SVG text editable in Illustrator
}
plt.rcParams.update(PUB_RC)

# ── Save helper ───────────────────────────────────────────────────────────────
def _save(fig, name, formats=("pdf", "png"), dpi=300):
    """Save figure in one or more publication formats."""
    # PDF for typesetting (vector, font-embedded), PNG for quick preview.
    os.makedirs(FIGURE_DIR, exist_ok=True)
    saved = []
    for fmt in formats:
        path = os.path.join(FIGURE_DIR, f"{name}.{fmt}")
        fig.savefig(path, format=fmt, dpi=dpi,
                    bbox_inches="tight", pad_inches=0.05)
        saved.append(path)
    print(f"  Saved: {', '.join(saved)}")

# ── Combined violin + box helper ──────────────────────────────────────────────
def violin_box(ax, data, x, y, palette, order=None):
    """
    Overlay a violin plot (distribution shape) with a narrow box plot
    (median + IQR + outliers) on the same axes.
    This is the recommended publication approach — more informative than
    either plot alone.
    """
    # Rationale: a box plot alone hides multimodality (two clusters with the
    # same median and IQR can have completely different shapes); a violin alone
    # makes the median and quartiles hard to read off precisely.
    sns.violinplot(
        x=x, y=y, data=data, ax=ax,
        palette=palette, order=order,
        inner=None,          # suppress inner box — we draw our own
        linewidth=0.8,
        alpha=0.55,
        saturation=0.85,
        cut=0,               # do not extend beyond data range
                             # (avoids implying values that were never observed,
                             #  which matters for bounded measures like BMI)
    )
    sns.boxplot(
        x=x, y=y, data=data, ax=ax,
        palette=palette, order=order,
        width=0.18,          # narrow box sits inside the violin
        linewidth=1.0,
        flierprops=dict(marker="o", markersize=3,
                        markerfacecolor="0.4", markeredgewidth=0.5),
        boxprops=dict(alpha=0.85),
        medianprops=dict(color="black", linewidth=1.5),
    )

# ── Cluster label helper ──────────────────────────────────────────────────────
# Populated after Section 9 subtype mapping runs; used for x-axis labels.
# Pre-define here so all sections can reference it safely.
# IMPORTANT: sections 4-8 execute BEFORE Section 9 fills this, so their figures
# fall back to plain cluster integers — see "Known issues" in the header.
CLUSTER_NAMES = {}   # filled in cell-024

def cluster_label(val):
    """Return 'SIDD (0)' style label for a cluster integer."""
    # Maps the verbose subtype names to compact abbreviations that fit on an
    # axis tick without overlapping.
    short = {'SIDD (Severe Insulin Deficient)': 'SIDD',
             'SOIRD (Severe Insulin Resistant)': 'SOIRD',
             'MARD (Mild Age-Related)': 'MARD',
             'MOD (Mild Obesity-Related)': 'MOD'}
    name = CLUSTER_NAMES.get(val, '')
    abbr = short.get(name, name)
    # Cluster number is retained alongside the name so a reader can always
    # trace a figure back to the numeric labels in the exported CSV.
    return f"{abbr}\n(C{val})" if abbr else f"C{val}"


def kw_dunn(df, value_col, group_col='Consensus_Cluster'):
    """
    Run Kruskal-Wallis test and pairwise Dunn post-hoc
    (Bonferroni correction) for a continuous variable.
    Returns (H, p_kw, dunn_df).
    """
    # Non-parametric throughout: the clinical variables here (HOMA indices,
    # triglycerides, diabetes duration) are markedly skewed, so rank-based
    # tests are more appropriate than ANOVA/t-tests.
    groups = [g[value_col].dropna().values
              for _, g in df.groupby(group_col)]
    H, p_kw = kruskal(*groups)   # omnibus: "do the groups differ anywhere?"

    cluster_ids = sorted(df[group_col].unique())
    pairs = list(combinations(cluster_ids, 2))
    n_pairs = len(pairs)          # Bonferroni denominator
    rows = []
    for c1, c2 in pairs:
        g1 = df[df[group_col] == c1][value_col].dropna().values
        g2 = df[df[group_col] == c2][value_col].dropna().values
        from scipy.stats import mannwhitneyu
        _, p_raw = mannwhitneyu(g1, g2, alternative='two-sided')
        # Bonferroni is the conservative choice; with only k(k-1)/2 comparisons
        # the power cost is small. min(..., 1.0) keeps it a valid probability.
        p_adj = min(p_raw * n_pairs, 1.0)   # Bonferroni
        rows.append({'C1': c1, 'C2': c2,
                     'p_raw': round(p_raw, 4),
                     'p_adj': round(p_adj, 4),
                     'sig': '***' if p_adj < 0.001 else
                            '**'  if p_adj < 0.01  else
                            '*'   if p_adj < 0.05  else 'ns'})
    return H, p_kw, pd.DataFrame(rows)


def annotate_kw(ax, df, value_col, group_col='Consensus_Cluster'):
    """Print Kruskal-Wallis result and add significance bracket to ax."""
    # Appends to the EXISTING title rather than replacing it, so the caller
    # keeps control of the descriptive part of the label.
    H, p, dunn = kw_dunn(df, value_col, group_col)
    p_str = '<0.001' if p < 0.001 else f'{p:.3f}'
    ax.set_title(ax.get_title() +
                 f'\nKW: H={H:.1f}, p={p_str}', fontsize=9)
    return H, p, dunn


print("Imports and helpers ready.")

## 2 · Load Data

In [ ]:
# FIX: corrected filename from k2_clusters_consensus.csv → consensus_k3_clusters.csv
# Input is the export from the clustering notebook: the analytical cohort plus
# Consensus_Cluster and the per-algorithm label columns.
CSV_FILE = "consensus_k3_clusters.csv"
df = pd.read_csv(CSV_FILE)

# Immediate sanity check — the cluster list confirms K matches the filename,
# which catches the case where an older CSV is still sitting in the folder.
print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
print(f"Clusters: {sorted(df['Consensus_Cluster'].unique())}")
df.head()

## 3 · Diabetes Status Distribution

NHANES diabetes coding: 1 = Yes, 2 = No, 3 = Borderline / Pre-diabetes,
9 = Refused / Don't know.

In [ ]:
# ── Validate column exists ────────────────────────────────────────────────────
# Fail loudly and early with the available columns listed, rather than raising
# an opaque KeyError deep inside a groupby further down.
if "Diabetes" not in df.columns:
    raise KeyError("Column 'Diabetes' not found. Available: " + str(list(df.columns)))

# NHANES DIQ010 coding. Note the cohort was already restricted to diabetes
# upstream, so codes other than 1.0 here reflect participants who met a
# laboratory or medication criterion without a self-reported diagnosis —
# which is exactly what this section is designed to show.
DIABETES_LABELS = {
    1.0: "Diabetes (Yes)",
    2.0: "No Diabetes",
    3.0: "Borderline / Pre-diabetes",
    9.0: "Unknown / Refused",
}

# FIX: map codes to readable strings before grouping — avoids float label issues
# fillna(astype(str)) preserves any unexpected code as its literal value rather
# than silently dropping those rows from the table.
df["Diabetes_Label"] = df["Diabetes"].map(DIABETES_LABELS).fillna(
    df["Diabetes"].astype(str)
)

# unstack turns the second grouping level into columns, giving a
# clusters x categories contingency table.
cluster_diabetes = (
    df.groupby(["Consensus_Cluster", "Diabetes_Label"])
    .size()
    .unstack(fill_value=0)   # absent combinations become 0, not NaN
)

print("Diabetes counts per cluster:")
print(cluster_diabetes)

In [ ]:
unique_clusters = sorted(df["Consensus_Cluster"].unique())
n_clusters      = len(unique_clusters)

# FIX: always ensure axes is a list — plt.subplots returns a single Axes
# object (not an array) when n_clusters == 1
fig, axes = plt.subplots(1, n_clusters,
                          figsize=(5.5 * n_clusters, 5))   # width scales with cluster count
axes = np.atleast_1d(axes)  # guarantee iterable regardless of n_clusters

# Consistent colour palette across all pies
# Building the palette ONCE outside the loop is what keeps a given diabetes
# category the same colour in every panel — otherwise each pie would recolour
# from its own first wedge.
N_CODES   = len(cluster_diabetes.columns)
PIE_CMAP  = plt.colormaps["Pastel1"].resampled(max(N_CODES, 3))
PIE_COLORS = [PIE_CMAP(i) for i in range(N_CODES)]

for i, cluster in enumerate(unique_clusters):
    data   = cluster_diabetes.loc[cluster]
    data   = data[data > 0]          # drop zero-count categories
                                     # (a 0% wedge would clutter the legend)
    colors = PIE_COLORS[:len(data)]

    wedges, texts, autotexts = axes[i].pie(
        data,
        labels=data.index,
        autopct="%1.1f%%",
        startangle=140,              # rotate so the largest wedge is not split by the top edge
        colors=colors,
        pctdistance=0.82,            # pull percentages inward, clear of the labels
        wedgeprops=dict(linewidth=0.6, edgecolor="white"),
    )
    for at in autotexts:
        at.set_fontsize(8)
    for t in texts:
        t.set_fontsize(8.5)

    n_total = data.sum()
    # Report n per panel: percentages alone hide very different cluster sizes.
    axes[i].set_title(f"Cluster {cluster}\n(n={n_total})", pad=10)

fig.suptitle("Diabetes Status Distribution by Cluster",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
_save(fig, "fig01_diabetes_pie")
plt.show()

## 4 · HOMA-IR & HOMA-B Classification

In [ ]:
# Clinical cut-points for insulin resistance. These are the conventional
# HOMA-IR bands; note they are population-dependent and no single set is
# universally agreed, so state the source in the manuscript methods.
def categorize_homa_ir(val):
    if pd.isna(val): return "Unknown"          # explicit category, not dropped
    if val < 1.0:    return "Insulin Sensitive (< 1.0)"
    if val < 2.0:    return "Normal / Borderline (1.0 – 1.9)"
    if val <= 2.9:   return "Moderate IR (2.0 – 2.9)"
    return "Severe IR (> 2.9)"

# HOMA-B expresses beta-cell function as a percentage of a normal reference,
# so 100% is "normal"; below 70% indicates deficiency, above 150% the
# compensatory hypersecretion typical of insulin-resistant phenotypes.
def categorize_homa_b(val):
    if pd.isna(val):  return "Unknown"
    if val < 70:      return "Low function (< 70%)"
    if val <= 150:    return "Normal / Compensatory (70 – 150%)"
    return "Hypersecretion (> 150%)"

df["HOMA_IR_Category"] = df["HOMA_IR"].apply(categorize_homa_ir)
df["HOMA_B_Category"]  = df["HOMA_B"].apply(categorize_homa_b)

# Explicit orderings so the stacked bars run from best to worst rather than
# alphabetically, which would scramble the clinical gradient.
IR_ORDER = [
    "Insulin Sensitive (< 1.0)",
    "Normal / Borderline (1.0 – 1.9)",
    "Moderate IR (2.0 – 2.9)",
    "Severe IR (> 2.9)",
]
B_ORDER = [
    "Low function (< 70%)",
    "Normal / Compensatory (70 – 150%)",
    "Hypersecretion (> 150%)",
]

# Row-normalise (axis=1): each CLUSTER sums to 100%, answering "what is the
# composition of this cluster?" rather than "how is this category split?".
homa_ir_counts = df.groupby(["Consensus_Cluster","HOMA_IR_Category"]).size().unstack(fill_value=0)
homa_ir_props  = homa_ir_counts.div(homa_ir_counts.sum(axis=1), axis=0) * 100
# Reindex to the clinical ordering, keeping only categories actually present.
homa_ir_props  = homa_ir_props[[c for c in IR_ORDER if c in homa_ir_props.columns]]

homa_b_counts  = df.groupby(["Consensus_Cluster","HOMA_B_Category"]).size().unstack(fill_value=0)
homa_b_props   = homa_b_counts.div(homa_b_counts.sum(axis=1), axis=0) * 100
homa_b_props   = homa_b_props[[c for c in B_ORDER if c in homa_b_props.columns]]

print("HOMA-IR Category Distribution (%):")
print(homa_ir_props.round(2))
print("\nHOMA-B Category Distribution (%):")
print(homa_b_props.round(2))

In [ ]:
# ── Map cluster integers to subtype labels for x-axis ───────────────────
# NOTE: CLUSTER_NAMES is still empty at this point (it is populated in
# Section 9), so this falls back to plain cluster numbers. See Known issues.
df['Cluster_Label'] = df['Consensus_Cluster'].apply(
    lambda x: cluster_label(x) if CLUSTER_NAMES else str(x)
)
C_ORDER = sorted(df['Consensus_Cluster'].unique())
C_LABELS = [cluster_label(c) if CLUSTER_NAMES else str(c) for c in C_ORDER]

# 2 rows (HOMA-IR, HOMA-B) x 3 views (composition, distribution, log scale).
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── Row 0 : HOMA-IR ────────────────────────────────────────────────────────
# Stacked bar
# Sequential "Reds" matches the severity gradient of the IR categories.
homa_ir_props.plot(kind="bar", stacked=True, ax=axes[0, 0],
                   colormap="Reds", edgecolor="white", linewidth=0.5)
axes[0, 0].set_title("HOMA-IR Classification by Cluster")
axes[0, 0].set_ylabel("Percentage (%)")
axes[0, 0].set_xlabel("Cluster")
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)
axes[0, 0].legend(title="HOMA-IR Status", bbox_to_anchor=(1.02, 1),
                   loc="upper left", fontsize=8)   # legend outside so it never covers a bar
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))

# FIX: Combined violin + box (replaces separate boxplot / violinplot)
violin_box(axes[0, 1], df, x="Cluster_Label", y="HOMA_IR", palette="Reds")
axes[0, 1].set_title("HOMA-IR Distribution")
axes[0, 1].set_xlabel("Cluster")
axes[0, 1].set_ylabel("HOMA-IR")
# Reference line at the conventional IR threshold, so cluster position can be
# read against a clinical anchor rather than only against the other clusters.
axes[0, 1].axhline(2.0, color="crimson", lw=1.0, ls="--", alpha=0.7,
                    label="IR threshold (2.0)")
axes[0, 1].legend(fontsize=8)

# Log-scale violin+box (HOMA-IR is right-skewed)
# On the linear axis a few extreme values compress every cluster into the
# bottom of the panel; the log view is what actually reveals the separation.
violin_box(axes[0, 2], df, x="Cluster_Label", y="HOMA_IR", palette="Reds")
axes[0, 2].set_yscale("log")
axes[0, 2].set_title("HOMA-IR Distribution (log scale)")
axes[0, 2].set_xlabel("Cluster")
axes[0, 2].set_ylabel("HOMA-IR (log)")

# ── Row 1 : HOMA-B ─────────────────────────────────────────────────────────
homa_b_props.plot(kind="bar", stacked=True, ax=axes[1, 0],
                  colormap="Blues", edgecolor="white", linewidth=0.5)
axes[1, 0].set_title("HOMA-B Classification by Cluster")
axes[1, 0].set_ylabel("Percentage (%)")
axes[1, 0].set_xlabel("Cluster")
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)
axes[1, 0].legend(title="HOMA-B Status", bbox_to_anchor=(1.02, 1),
                   loc="upper left", fontsize=8)
axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))

violin_box(axes[1, 1], df, x="Cluster_Label", y="HOMA_B", palette="Blues")
axes[1, 1].set_title("HOMA-B Distribution")
axes[1, 1].set_xlabel("Cluster")
axes[1, 1].set_ylabel("HOMA-B (%)")
# Two lines bracket the "normal / compensatory" band, so a cluster sitting
# below 70 reads immediately as beta-cell deficient (the SIDD signature).
axes[1, 1].axhline(70,  color="steelblue", lw=1.0, ls="--", alpha=0.7,
                    label="Low threshold (70)")
axes[1, 1].axhline(150, color="navy",      lw=1.0, ls="--", alpha=0.7,
                    label="Hyper threshold (150)")
axes[1, 1].legend(fontsize=8)

violin_box(axes[1, 2], df, x="Cluster_Label", y="HOMA_B", palette="Blues")
axes[1, 2].set_yscale("log")
axes[1, 2].set_title("HOMA-B Distribution (log scale)")
axes[1, 2].set_xlabel("Cluster")
axes[1, 2].set_ylabel("HOMA-B (log)")

fig.suptitle("HOMA-IR & HOMA-B Analysis by Cluster",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
_save(fig, "fig02_homa_classification")
plt.show()

## 5 · BMI & Age Analysis

In [ ]:
# Standard WHO BMI categories.
def categorize_bmi(val):
    if pd.isna(val): return "Unknown"
    if val < 18.5:   return "Underweight (< 18.5)"
    if val < 25:     return "Normal (18.5 – 24.9)"
    if val < 30:     return "Overweight (25 – 29.9)"
    return "Obese (≥ 30)"

# Age bands chosen to align with how diabetes phenotypes are usually described:
# MARD is defined by late onset, so separating 60-74 from 75+ matters here.
def categorize_age(val):
    if pd.isna(val): return "Unknown"
    if val < 40:     return "Young Adult (< 40)"
    if val <= 59:    return "Middle-Aged (40 – 59)"
    if val <= 74:    return "Older Adult (60 – 74)"
    return "Elderly (≥ 75)"

df["BMI_Category"] = df["BMI"].apply(categorize_bmi)
df["Age_Category"] = df["Age"].apply(categorize_age)

# Explicit orderings so bars run low-to-high rather than alphabetically.
BMI_ORDER = [
    "Underweight (< 18.5)", "Normal (18.5 – 24.9)",
    "Overweight (25 – 29.9)", "Obese (≥ 30)"
]
AGE_ORDER = [
    "Young Adult (< 40)", "Middle-Aged (40 – 59)",
    "Older Adult (60 – 74)", "Elderly (≥ 75)"
]

# Row-normalised: each cluster sums to 100% (composition within cluster).
bmi_counts = df.groupby(["Consensus_Cluster","BMI_Category"]).size().unstack(fill_value=0)
bmi_props  = bmi_counts.div(bmi_counts.sum(axis=1), axis=0) * 100
bmi_props  = bmi_props[[c for c in BMI_ORDER if c in bmi_props.columns]]

age_counts = df.groupby(["Consensus_Cluster","Age_Category"]).size().unstack(fill_value=0)
age_props  = age_counts.div(age_counts.sum(axis=1), axis=0) * 100
age_props  = age_props[[c for c in AGE_ORDER if c in age_props.columns]]

print("BMI Category Distribution (%):")
print(bmi_props.round(2))
print("\nAge Category Distribution (%):")
print(age_props.round(2))

In [ ]:
# ── Map cluster integers to subtype labels for x-axis ───────────────────
# CLUSTER_NAMES is still empty here (populated in Section 9), so this falls
# back to cluster numbers — see Known issues in the header.
df['Cluster_Label'] = df['Consensus_Cluster'].apply(
    lambda x: cluster_label(x) if CLUSTER_NAMES else str(x)
)
C_ORDER = sorted(df['Consensus_Cluster'].unique())
C_LABELS = [cluster_label(c) if CLUSTER_NAMES else str(c) for c in C_ORDER]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── Row 0 : BMI ────────────────────────────────────────────────────────────
bmi_props.plot(kind="bar", stacked=True, ax=axes[0, 0],
               colormap="coolwarm", edgecolor="white", linewidth=0.5)
axes[0, 0].set_title("BMI Classification by Cluster")
axes[0, 0].set_ylabel("Percentage (%)")
axes[0, 0].set_xlabel("Cluster")
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)
axes[0, 0].legend(title="BMI Category", bbox_to_anchor=(1.02, 1),
                   loc="upper left", fontsize=8)
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))

# FIX: combined violin + box replaces separate plots
violin_box(axes[0, 1], df, x="Cluster_Label", y="BMI", palette="coolwarm")
axes[0, 1].set_title("BMI Distribution")
axes[0, 1].set_xlabel("Cluster")
axes[0, 1].set_ylabel("BMI (kg/m²)")
# Clinical anchors: 30 is the obesity threshold that separates MOD/SOIRD-type
# phenotypes from MARD in the Ahlqvist framework.
axes[0, 1].axhline(25, color="orange", lw=1.0, ls="--", alpha=0.7,
                    label="Overweight (25)")
axes[0, 1].axhline(30, color="crimson", lw=1.0, ls="--", alpha=0.7,
                    label="Obese (30)")
axes[0, 1].legend(fontsize=8)

# Swarm overlay on the violin+box for smaller datasets
# The stripplot shows actual sample density behind the smoothed violin;
# alpha 0.25 + small markers keep it readable at n in the thousands, and
# rasterized=True stops the PDF ballooning with thousands of vector points.
violin_box(axes[0, 2], df, x="Cluster_Label", y="BMI", palette="coolwarm")
sns.stripplot(x="Cluster_Label", y="BMI", data=df, ax=axes[0, 2],
              size=2.5, color="0.3", alpha=0.25, jitter=True, rasterized=True)
axes[0, 2].set_title("BMI Distribution (with data points)")
axes[0, 2].set_xlabel("Cluster")
axes[0, 2].set_ylabel("BMI (kg/m²)")

# ── Row 1 : Age ────────────────────────────────────────────────────────────
age_props.plot(kind="bar", stacked=True, ax=axes[1, 0],
               colormap="Oranges", edgecolor="white", linewidth=0.5)
axes[1, 0].set_title("Age Group Classification by Cluster")
axes[1, 0].set_ylabel("Percentage (%)")
axes[1, 0].set_xlabel("Cluster")
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)
axes[1, 0].legend(title="Age Group", bbox_to_anchor=(1.02, 1),
                   loc="upper left", fontsize=8)
axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))

violin_box(axes[1, 1], df, x="Cluster_Label", y="Age", palette="Oranges")
axes[1, 1].set_title("Age Distribution")
axes[1, 1].set_xlabel("Cluster")
axes[1, 1].set_ylabel("Age (years)")
axes[1, 1].axhline(40, color="orange", lw=1.0, ls="--", alpha=0.7,
                    label="Middle-age threshold (40)")
axes[1, 1].axhline(60, color="darkorange", lw=1.0, ls="--", alpha=0.7,
                    label="Older-adult threshold (60)")
axes[1, 1].legend(fontsize=8)

violin_box(axes[1, 2], df, x="Cluster_Label", y="Age", palette="Oranges")
sns.stripplot(x="Cluster_Label", y="Age", data=df, ax=axes[1, 2],
              size=2.5, color="0.3", alpha=0.25, jitter=True, rasterized=True)
axes[1, 2].set_title("Age Distribution (with data points)")
axes[1, 2].set_xlabel("Cluster")
axes[1, 2].set_ylabel("Age (years)")

fig.suptitle("BMI & Age Analysis by Cluster",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
_save(fig, "fig03_bmi_age_analysis")
plt.show()

## 6 · Sex Distribution

In [ ]:
if "Sex" not in df.columns:
    raise KeyError("Column 'Sex' not found. Available: " + str(list(df.columns)))

# NHANES RIAGENDR: 1 = Male, 2 = Female.
SEX_LABELS  = {1.0: "Male", 2.0: "Female"}
SEX_COLORS  = ["#4C72B0", "#DD8452"]   # blue, orange — colourblind-safe

# fillna(astype(str)) keeps any unexpected code visible instead of dropping it.
df["Sex_Label"] = df["Sex"].map(SEX_LABELS).fillna(df["Sex"].astype(str))

unique_clusters = sorted(df["Consensus_Cluster"].unique())
n_clusters      = len(unique_clusters)

# FIX: np.atleast_1d ensures axes is always iterable for any n_clusters
fig, axes = plt.subplots(1, n_clusters,
                          figsize=(5 * n_clusters, 5))
axes = np.atleast_1d(axes)

for i, cluster in enumerate(unique_clusters):
    data   = df[df["Consensus_Cluster"] == cluster]["Sex_Label"].value_counts()
    # CAUTION: value_counts() orders by FREQUENCY, so if the majority sex
    # differs between clusters the blue/orange assignment can flip between
    # panels. Passing an explicit order would pin it.
    colors = SEX_COLORS[:len(data)]

    wedges, texts, autotexts = axes[i].pie(
        data,
        labels=data.index,
        autopct="%1.1f%%",
        startangle=90,               # first wedge starts at 12 o'clock
        colors=colors,
        pctdistance=0.80,
        wedgeprops=dict(linewidth=0.6, edgecolor="white"),
    )
    for at in autotexts:
        at.set_fontsize(9)
    axes[i].set_title(f"Cluster {cluster}\n(n={data.sum()})", pad=8)

fig.suptitle("Sex Distribution by Cluster",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
_save(fig, "fig04_sex_distribution")
plt.show()

## 7 · eGFR / Kidney Health (CKD Staging)

In [ ]:
# Whole section is guarded: eGFR is derived upstream and may be absent if an
# older cohort file is loaded, so warn and skip rather than crash the notebook.
if "eGFR" not in df.columns:
    print("WARNING: 'eGFR' column not found — skipping this section.")
    print("Available columns:", list(df.columns))
else:
    # KDIGO chronic kidney disease staging by eGFR. Stage 3 is split into
    # 3a/3b because that boundary carries different management implications.
    def categorize_egfr(val):
        if pd.isna(val):  return "Unknown"
        if val >= 90:     return "Stage 1: Normal (≥ 90)"
        if val >= 60:     return "Stage 2: Mild (60 – 89)"
        if val >= 45:     return "Stage 3a: Mild-Mod (45 – 59)"
        if val >= 30:     return "Stage 3b: Mod-Severe (30 – 44)"
        if val >= 15:     return "Stage 4: Severe (15 – 29)"
        return "Stage 5: Kidney Failure (< 15)"

    # Ordered best-to-worst so the stacked bar reads as a severity gradient.
    EGFR_ORDER = [
        "Stage 1: Normal (≥ 90)",
        "Stage 2: Mild (60 – 89)",
        "Stage 3a: Mild-Mod (45 – 59)",
        "Stage 3b: Mod-Severe (30 – 44)",
        "Stage 4: Severe (15 – 29)",
        "Stage 5: Kidney Failure (< 15)",
    ]

    df["eGFR_Category"] = df["eGFR"].apply(categorize_egfr)

    # Row-normalised: composition within each cluster.
    egfr_counts = df.groupby(["Consensus_Cluster","eGFR_Category"]).size().unstack(fill_value=0)
    egfr_props  = egfr_counts.div(egfr_counts.sum(axis=1), axis=0) * 100
    egfr_props  = egfr_props[[c for c in EGFR_ORDER if c in egfr_props.columns]]

    print("eGFR Category Distribution (%):")
    print(egfr_props.round(2))

In [ ]:
# ── Map cluster integers to subtype labels for x-axis ───────────────────
# Still before Section 9, so labels fall back to cluster numbers.
df['Cluster_Label'] = df['Consensus_Cluster'].apply(
    lambda x: cluster_label(x) if CLUSTER_NAMES else str(x)
)
C_ORDER = sorted(df['Consensus_Cluster'].unique())
C_LABELS = [cluster_label(c) if CLUSTER_NAMES else str(c) for c in C_ORDER]

if "eGFR" in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Combined violin + box
    violin_box(axes[0], df, x="Cluster_Label", y="eGFR", palette="Set2")
    axes[0].set_title("eGFR Distribution (Violin + Box)")
    axes[0].set_ylabel("eGFR (mL/min/1.73 m²)")
    axes[0].set_xlabel("Cluster")
    # 60 is the diagnostic threshold for CKD; 90 the lower bound of normal.
    # Showing both lets the reader judge the shift toward impairment directly.
    axes[0].axhline(60, color="crimson", lw=1.2, ls="--", alpha=0.8,
                     label="CKD threshold (60)")
    axes[0].axhline(90, color="steelblue", lw=1.0, ls=":", alpha=0.7,
                     label="Normal lower limit (90)")
    axes[0].legend(fontsize=8)

    # With data points
    violin_box(axes[1], df, x="Cluster_Label", y="eGFR", palette="Set2")
    sns.stripplot(x="Cluster_Label", y="eGFR", data=df, ax=axes[1],
                  size=2.5, color="0.3", alpha=0.25, jitter=True, rasterized=True)
    axes[1].set_title("eGFR Distribution (with data points)")
    axes[1].set_ylabel("eGFR (mL/min/1.73 m²)")
    axes[1].set_xlabel("Cluster")
    axes[1].axhline(60, color="crimson", lw=1.2, ls="--", alpha=0.8,
                     label="CKD threshold (60)")
    axes[1].legend(fontsize=8)

    # Stacked CKD-stage bar
    # FIX: use reversed RdYlGn so red = severe, green = normal
    # (the default RdYlGn would colour kidney failure green — actively misleading)
    egfr_props.plot(kind="bar", stacked=True, ax=axes[2],
                    colormap="RdYlGn_r", edgecolor="white", linewidth=0.5)
    axes[2].set_title("Kidney Health — CKD Stages by Cluster")
    axes[2].set_ylabel("Percentage (%)")
    axes[2].set_xlabel("Cluster")
    axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
    axes[2].legend(title="CKD Stage", bbox_to_anchor=(1.02, 1),
                    loc="upper left", fontsize=8)
    axes[2].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda y, _: f"{y:.0f}%")
    )

    fig.suptitle("eGFR / Kidney Health by Cluster",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    _save(fig, "fig05_egfr_analysis")
    plt.show()

## 8 · HbA1c Classification

In [ ]:
# ADA diagnostic and control thresholds. Note the cohort is already all
# diabetes, so the "Normal" and "Prediabetes" bands capture participants
# identified by medication or glucose criteria rather than by HbA1c —
# a useful check on how ascertainment differs between clusters.
def categorize_hba1c(val):
    if pd.isna(val): return "Unknown"
    if val < 5.7:    return "Normal (< 5.7%)"
    if val < 6.5:    return "Prediabetes (5.7 – 6.4%)"
    if val < 8.0:    return "Diabetes – Controlled (6.5 – 7.9%)"
    return "Poor Control (≥ 8.0%)"

HBA1C_ORDER = [
    "Normal (< 5.7%)",
    "Prediabetes (5.7 – 6.4%)",
    "Diabetes – Controlled (6.5 – 7.9%)",
    "Poor Control (≥ 8.0%)",
]

df["HbA1c_Category"] = df["HbA1c"].apply(categorize_hba1c)

hba1c_counts = df.groupby(["Consensus_Cluster","HbA1c_Category"]).size().unstack(fill_value=0)
hba1c_props  = hba1c_counts.div(hba1c_counts.sum(axis=1), axis=0) * 100
hba1c_props  = hba1c_props[[c for c in HBA1C_ORDER if c in hba1c_props.columns]]

print("HbA1c Category Distribution (%):")
print(hba1c_props.round(2))

In [ ]:
# ── Map cluster integers to subtype labels for x-axis ───────────────────
# Last section before the subtype mapping, so still numeric labels.
df['Cluster_Label'] = df['Consensus_Cluster'].apply(
    lambda x: cluster_label(x) if CLUSTER_NAMES else str(x)
)
C_ORDER = sorted(df['Consensus_Cluster'].unique())
C_LABELS = [cluster_label(c) if CLUSTER_NAMES else str(c) for c in C_ORDER]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Stacked bar
hba1c_props.plot(kind="bar", stacked=True, ax=axes[0],
                 colormap="Purples", edgecolor="white", linewidth=0.5)
axes[0].set_title("HbA1c Classification by Cluster")
axes[0].set_ylabel("Percentage (%)")
axes[0].set_xlabel("Cluster")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title="HbA1c Status", bbox_to_anchor=(1.02, 1),
                loc="upper left", fontsize=8)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))

# Combined violin + box
violin_box(axes[1], df, x="Cluster_Label", y="HbA1c", palette="Purples")
axes[1].set_title("HbA1c Distribution")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("HbA1c (%)")
# Three anchors: diagnosis (6.5) and the poor-control boundary (8.0) are the
# two that matter clinically; 5.7 is included for completeness.
axes[1].axhline(5.7, color="orange",  lw=1.0, ls="--", alpha=0.8,
                 label="Prediabetes (5.7%)")
axes[1].axhline(6.5, color="crimson", lw=1.0, ls="--", alpha=0.8,
                 label="Diabetes (6.5%)")
axes[1].axhline(8.0, color="darkred", lw=1.0, ls="--", alpha=0.8,
                 label="Poor control (8.0%)")
axes[1].legend(fontsize=8)

# With data points
violin_box(axes[2], df, x="Cluster_Label", y="HbA1c", palette="Purples")
sns.stripplot(x="Cluster_Label", y="HbA1c", data=df, ax=axes[2],
              size=2.5, color="0.3", alpha=0.25, jitter=True, rasterized=True)
axes[2].set_title("HbA1c Distribution (with data points)")
axes[2].set_xlabel("Cluster")
axes[2].set_ylabel("HbA1c (%)")

fig.suptitle("HbA1c Classification by Cluster",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
_save(fig, "fig06_hba1c_classification")
plt.show()

## 9 · Clinical Profiles & Ahlqvist Subtype Mapping

Maps each consensus cluster to an Ahlqvist et al. (2018) subtype
using a greedy priority assignment:

1. **SIDD** — lowest HOMA-B (severe insulin deficiency)
2. **SIRD** — highest HOMA-IR among remaining (severe insulin resistance)
3. **MOD / MARD** — remaining cluster: MOD if mean BMI ≥ 30, else MARD

In [ ]:
# The five clustering features, in the order used throughout this section.
FEATURES = ["Age", "BMI", "HbA1c", "HOMA_IR", "HOMA_B"]

# Cluster centroids on the ORIGINAL clinical scale. These means drive the
# Ahlqvist subtype assignment in the next cell.
# NOTE: the clustering itself used log-transformed HOMA values, so the means
# here are arithmetic means of skewed variables and sit above the medians.
profile = df.groupby("Consensus_Cluster")[FEATURES].mean()
print("Cluster mean profiles:")
print(profile.round(3))

In [ ]:
# FIX: renamed local dict from 'labels' (shadows common variable) to
# 'subtype_map' throughout this section
# Greedy, priority-ordered assignment: each step claims the most distinctive
# remaining cluster, then removes it so later steps cannot re-use it. Order
# matters — the most pathognomonic marker (beta-cell failure) is claimed first.
subtype_map = {}
remaining   = profile.copy()

# Step 1 — SIDD: lowest beta-cell function (HOMA-B)
# Severe insulin-deficient diabetes is defined primarily by beta-cell failure,
# which is the least ambiguous single-feature signature of the four subtypes.
sidd_id = remaining["HOMA_B"].idxmin()
subtype_map[sidd_id] = "SIDD (Severe Insulin Deficient Diabetes)"
remaining = remaining.drop(index=sidd_id)

if len(remaining) >= 1:
    # Step 2 — SIRD: highest insulin resistance (HOMA-IR) among remaining
    sird_id = remaining["HOMA_IR"].idxmax()
    subtype_map[sird_id] = "SOIRD (Severe Obesity-Related Insulin-Resistant Diabetes)"  # Severe Obesity+IR
    remaining = remaining.drop(index=sird_id)

# Step 3 — MOD / MARD: classify any leftover clusters
# The remaining cluster(s) are "mild" by elimination; BMI 30 then separates
# obesity-related (MOD) from age-related (MARD).
for final_id in remaining.index:
    bmi_mean = remaining.loc[final_id, "BMI"]
    if bmi_mean >= 30:
        subtype_map[final_id] = "MOD (Mild Obesity-Related Diabetes)"
    else:
        subtype_map[final_id] = "MARD (Mild Age-Related Diabetes)"

# Apply mapping
df["Subtype_Label"] = df["Consensus_Cluster"].map(subtype_map)

print("Ahlqvist subtype mapping:")
for k, v in sorted(subtype_map.items()):
    n = (df["Consensus_Cluster"] == k).sum()
    print(f"  Cluster {k} → {v}  (n={n})")

print()
print("Subtype counts:")
print(df["Subtype_Label"].value_counts().to_string())

# ── Populate CLUSTER_NAMES for axis labelling throughout notebook ───────────
# From this point on, cluster_label() returns subtype abbreviations. Sections
# 4-8 already ran with numeric labels — re-run them if consistent axis naming
# is needed across the whole figure set.
CLUSTER_NAMES.update(subtype_map)
print(f"\nCLUSTER_NAMES populated: {CLUSTER_NAMES}")

In [ ]:
# ── Z-score heatmap ──────────────────────────────────────────────────────────
# FIX: normalise row-wise (per feature across clusters) using ddof=0 to
# avoid NaN when only 2 clusters are present (std with ddof=1 → NaN for n=2)
# Standardising each FEATURE across the cluster means puts every feature on a
# common scale, so the heatmap shows relative position (which cluster is high
# on which axis) rather than raw magnitudes in incompatible units.
# NOTE: with only K cluster means, |z| is bounded by sqrt(K-1) — about 1.41 at
# K=3 — so the colour range is inherently compressed. Do not read the values
# as participant-level z-scores.
profile_norm = (profile - profile.mean()) / profile.std(ddof=0)

# Replace NaN columns (zero-variance features) with 0
# A feature identical across all clusters has std 0 → 0/0; 0 is the correct
# "no deviation" value.
profile_norm = profile_norm.fillna(0)

# Use subtype names as row index for readability
profile_norm.index = [
    subtype_map.get(i, f"Cluster {i}") for i in profile_norm.index
]

# Height scales with the number of clusters so cells stay roughly square.
fig, ax = plt.subplots(figsize=(9, max(3, len(profile_norm) * 1.2)))

sns.heatmap(
    profile_norm,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",     # diverging
    center=0,          # anchor white at the mean, so red/blue read as above/below
    linewidths=0.5,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Z-score", "fraction": 0.046, "pad": 0.04,
               "shrink": 0.8},
    annot_kws={"size": 9},
)

ax.set_title("Clinical Profiles of Consensus Clusters (Z-scores)", pad=12)
ax.set_xlabel("Clinical Feature")
ax.set_ylabel("Cluster / Subtype")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
_save(fig, "fig07_clinical_profiles_heatmap")
plt.show()

In [ ]:
# ── Radar / spider chart of mean profiles ────────────────────────────────────
# Normalise 0-1 for radar (min-max within each feature, across cluster means)
# Min-max rather than z-score here because a radar needs a fixed [0, 1] radial
# range; replace(0, 1) guards against divide-by-zero for a constant feature.
# CAVEAT: with min-max across only K clusters, the lowest cluster always sits
# at 0 and the highest at 1 on every axis, so the chart shows RANK/relative
# position, not effect size.
profile_radar = (profile - profile.min()) / (profile.max() - profile.min()).replace(0, 1)

# Typeset names (HOMA-IR rather than HOMA_IR) for the spoke labels.
FEAT_DISPLAY = {
    "Age": "Age",
    "BMI": "BMI",
    "HbA1c": "HbA1c",
    "HOMA_IR": "HOMA-IR",
    "HOMA_B": "HOMA-B",
}
feat_labels = [FEAT_DISPLAY.get(f, f.replace("_", "-")) for f in FEATURES]
N = len(feat_labels)
# Equally spaced angles around the circle, one spoke per feature.
angles  = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon
                       # (repeat the first angle so the line returns to start)

fig, ax = plt.subplots(figsize=(6.5, 7.5), subplot_kw=dict(polar=True))
cmap_r = plt.colormaps["tab10"].resampled(max(len(profile_radar), 2))

for idx_r, (cluster_id, row) in enumerate(profile_radar.iterrows()):
    values = row.tolist() + row.tolist()[:1]   # close the polygon to match angles
    label  = subtype_map.get(cluster_id, f"Cluster {cluster_id}")
    # Indexed by ENUMERATION order (0,1,2...), which avoids the out-of-range
    # clipping that indexing by cluster id would cause on a resampled colormap.
    color  = cmap_r(idx_r)
    ax.plot(angles, values, "o-", linewidth=1.8, color=color,
            markersize=5, label=label)
    ax.fill(angles, values, alpha=0.08, color=color)   # faint fill aids shape reading

ax.set_thetagrids(np.degrees(angles[:-1]), feat_labels, fontsize=10)
ax.tick_params(axis="x", pad=14)          # push spoke labels clear of the outer ring
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.50, 0.75])
ax.set_yticklabels(["0.25", "0.50", "0.75"], fontsize=7, color="0.5")
ax.set_rlabel_position(90 / N)            # park radial labels between two spokes
ax.grid(color="0.8", linewidth=0.5)

ax.set_title("Clinical Feature Radar NHANES 1999-2018", pad=28, fontsize=12, fontweight="bold")

# Legend below the plot: subtype names are long and would overlap the polygon
# if placed inside or beside it.
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08),
          ncol=1, fontsize=9, frameon=True, framealpha=0.9,
          borderaxespad=0.0)

fig.subplots_adjust(top=0.86, bottom=0.24)   # replaces tight_layout
                                             # (tight_layout mishandles polar axes
                                             #  with an external legend)
_save(fig, "fig08_clinical_radar")
plt.show()

## 10 · Statistical Tests — Kruskal-Wallis & Dunn Post-Hoc

Non-parametric Kruskal-Wallis H test is applied to all continuous clustering and clinical variables. Significant results are followed by pairwise Dunn tests with Bonferroni correction.

In [ ]:
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations

# Screens every continuous clinical variable, not just the five used for
# clustering. IMPORTANT INTERPRETIVE NOTE: the first five were the clustering
# inputs, so their significance is near-guaranteed and is not evidence the
# clusters are real. The variables NOT used for clustering (eGFR, BP, lipids,
# duration) are the informative ones — differences there are genuine external
# validation of the phenotypes.
CONTINUOUS = [
    'Age', 'BMI', 'HbA1c', 'HOMA_IR', 'HOMA_B',
    'eGFR', 'Systolic_BP', 'Diastolic_BP',
    'LDL', 'HDL', 'Triglycerides',
    'Fasting_glucose', 'Fasting_insulin', 'Diabetes_duration',
]

kw_results = []
for var in CONTINUOUS:
    if var not in df.columns:
        continue                      # skip gracefully if a variable is absent
    groups = [g[var].dropna().values
              for _, g in df.groupby('Consensus_Cluster')]
    # Kruskal-Wallis needs at least 2 observations per group to be meaningful.
    if any(len(g) < 2 for g in groups):
        continue
    H, p = kruskal(*groups)
    kw_results.append({'Variable': var,
                       'H': round(H, 2),
                       'p_value': round(p, 4),
                       'Significant': '***' if p < 0.001 else
                                      '**'  if p < 0.01  else
                                      '*'   if p < 0.05  else 'ns'})

# Sorted by p so the strongest discriminators appear first.
# CAUTION: no multiple-testing correction is applied ACROSS the 14 variables
# here; consider FDR if this table is reported as a screening result.
kw_df = pd.DataFrame(kw_results).sort_values('p_value')
print('Kruskal-Wallis results (all continuous variables):')
print(kw_df.to_string(index=False))

## 11 · Table 1 — Clinical Summary by Subtype

Publication-ready summary statistics (median [IQR] for continuous; n (%) for categorical) with Kruskal-Wallis p-values.

In [ ]:
from scipy.stats import kruskal, chi2_contingency

# NOTE: this is the FIRST of two Table 1 implementations. Section 11b builds a
# more complete version (harmonised categories, explicit missingness, Overall
# column). Decide which is canonical before submission — see Known issues.
CONT_VARS = [
    'Age', 'BMI', 'HbA1c', 'HOMA_IR', 'HOMA_B', 'eGFR',
    'Systolic_BP', 'Diastolic_BP', 'LDL', 'HDL',
    'Triglycerides', 'Fasting_glucose', 'Diabetes_duration',
]
CAT_VARS = [
    'Sex', 'Ethnicity', 'Education_level',
    'Smoking_category', 'Alcohol_status', 'Physical_activity',
    'HbA1c_Category', 'BMI_Category', 'eGFR_Category',
]

clusters = sorted(df['Consensus_Cluster'].unique())
# Column headers use the subtype names so the table is self-describing.
col_names = [subtype_map.get(c, f'Cluster {c}') for c in clusters]

rows = []

# ── Continuous variables: median [Q1, Q3] + KW p-value ───────────────────────
# Median/IQR rather than mean/SD because these variables are skewed; this also
# matches the non-parametric test used alongside them.
for var in CONT_VARS:
    if var not in df.columns:
        continue
    row = {'Variable': var, 'Type': 'continuous'}
    groups = []
    for c, cn in zip(clusters, col_names):
        vals = df[df['Consensus_Cluster'] == c][var].dropna()
        q1, med, q3 = vals.quantile([0.25, 0.50, 0.75])
        row[cn] = f'{med:.1f} [{q1:.1f}, {q3:.1f}]'
        groups.append(vals.values)
    if all(len(g) >= 2 for g in groups):
        _, p = kruskal(*groups)
        # '<0.001' rather than a rounded 0.000, which would read as exactly zero.
        row['p_value'] = '<0.001' if p < 0.001 else f'{p:.3f}'
    else:
        row['p_value'] = 'N/A'
    rows.append(row)

# ── Categorical variables: n (%) + chi-square p-value ────────────────────────
for var in CAT_VARS:
    if var not in df.columns:
        continue
    # Overall chi-square
    # One omnibus test per variable, reported on the header row; the level rows
    # below carry no individual p-values.
    ct = pd.crosstab(df['Consensus_Cluster'], df[var])
    try:
        chi2, p_chi, _, _ = chi2_contingency(ct)
        p_str = '<0.001' if p_chi < 0.001 else f'{p_chi:.3f}'
    except Exception:
        # Chi-square fails on degenerate tables (a single level, or all-zero rows).
        p_str = 'N/A'

    # Header row for this variable
    hrow = {'Variable': var, 'Type': 'categorical_header', 'p_value': p_str}
    for cn in col_names:
        hrow[cn] = ''            # blank cells: the numbers live on the level rows
    rows.append(hrow)

    # One row per category level
    for level in df[var].dropna().unique():
        lrow = {'Variable': f'  {level}', 'Type': 'categorical_level', 'p_value': ''}
        for c, cn in zip(clusters, col_names):
            sub = df[df['Consensus_Cluster'] == c]
            n   = (sub[var] == level).sum()
            # Percentage denominator is the FULL cluster size, so levels within
            # a cluster sum to 100% only if the variable has no missing values.
            pct = 100 * n / len(sub)
            lrow[cn] = f'{n} ({pct:.1f}%)'
        rows.append(lrow)

table1 = pd.DataFrame(rows)
# Reorder to the display sequence, keeping only columns that were actually built.
display_cols = ['Variable'] + col_names + ['p_value']
display_cols = [c for c in display_cols if c in table1.columns]
table1 = table1[display_cols]

print('TABLE 1 — Clinical Characteristics by Subtype')
print('=' * 100)
print(table1.to_string(index=False))

# Save to CSV for manuscript
table1.to_csv('table1_clinical_summary.csv', index=False)
print('\nSaved: table1_clinical_summary.csv')

## 11b · Comprehensive Baseline Characteristics — Publication Table 1

Baseline characteristics of participants **stratified by the identified
consensus clusters (Ahlqvist subtypes)** across **all continuous and
categorical variables**. Missing data are reported explicitly.

| Variable | NHANES source | Levels used here |
|---|---|---|
| Sex | `RIAGENDR` | Male, Female |
| Race/Ethnicity | `RIDRETH1` | Mexican American, Other Hispanic, Non-Hispanic White, Non-Hispanic Black, Other Race |
| Education level | `DMDEDUC2` | Less than high school (codes 1–2), High school (code 3), More than high school (codes 4–5) |
| Family PIR | `INDFMPIR` | Low (< 1.3), Medium (1.3 – < 3.5), High (≥ 3.5) |

**Continuous** — median [Q1, Q3]; Kruskal–Wallis *p*.
**Categorical** — n (%); χ² *p* (computed over non-missing cells).


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11b.1 — Harmonise categorical variables to NHANES-official labels
# ══════════════════════════════════════════════════════════════════════
# Why this step exists: NHANES codes these variables numerically and the coding
# changed across cycles, so a raw crosstab would produce meaningless float
# categories. Each helper below accepts EITHER a numeric code or an already-
# decoded string, so the notebook works whichever upstream version produced
# the CSV.
import numpy as np
import pandas as pd

# Work on a copy: the harmonised *_Cat columns are for Table 1 only and should
# not leak into the exported dataset.
df_t1 = df.copy()

def sex_category(v):
    if pd.isna(v): return np.nan
    try:
        code = int(float(v))
    except (TypeError, ValueError):
        # Already a string: match on first letter so 'M'/'Male'/'male' all work.
        s = str(v).strip().lower()
        return 'Male' if s.startswith('m') else 'Female' if s.startswith('f') else np.nan
    return {1: 'Male', 2: 'Female'}.get(code, np.nan)

if 'Sex' in df_t1.columns:
    df_t1['Sex_Cat'] = df_t1['Sex'].apply(sex_category)

# Family PIR: Low < 1.3, Medium 1.3–<3.5, High >= 3.5
# PIR is income as a ratio to the federal poverty line; 1.3 and 3.5 are the
# conventional NHANES cut-points (they align with programme eligibility bands).
def pir_category(v):
    if pd.isna(v): return np.nan
    try: v = float(v)
    except (TypeError, ValueError):
        s = str(v).strip().lower()
        if s.startswith('low'):  return 'Low (PIR < 1.3)'
        if s.startswith('med') or s.startswith('mid'): return 'Medium (1.3 \u2264 PIR < 3.5)'
        if s.startswith('high'): return 'High (PIR \u2265 3.5)'
        return np.nan
    if v < 1.3: return 'Low (PIR < 1.3)'
    if v < 3.5: return 'Medium (1.3 \u2264 PIR < 3.5)'
    return 'High (PIR \u2265 3.5)'

# Column name for PIR varies by upstream version — take the first that exists.
_PIR_COLS = ['Family_PIR', 'INDFMPIR', 'PIR', 'family_pir', 'pir']
pir_col = next((c for c in _PIR_COLS if c in df_t1.columns), None)
if pir_col:
    df_t1['Family_PIR_Cat'] = df_t1[pir_col].apply(pir_category)
    print(f'  Family_PIR source: {pir_col!r}')
else:
    print('  Family_PIR: NOT FOUND')

# Education: DMDEDUC2
# Collapsed from five levels to three: the original categories are too sparse
# per cluster to support a stable chi-square. Codes 7/9 (Refused/Don't know)
# map to NaN so they are reported as missing rather than as a real category.
_EDU_NUM = {1: 'Less than high school', 2: 'Less than high school',
            3: 'High school',
            4: 'More than high school', 5: 'More than high school',
            7: np.nan, 9: np.nan}
# String fallback covers the various label spellings NHANES has used.
_EDU_STR = {
    'less than 9th grade':                                'Less than high school',
    '9-11th grade':                                       'Less than high school',
    '9-11th grade (includes 12th grade with no diploma)': 'Less than high school',
    'high school graduate/ged or equivalent':             'High school',
    'high school grad/ged or equivalent':                 'High school',
    'high school':                                        'High school',
    'some college or aa degree':                          'More than high school',
    'college graduate or above':                          'More than high school',
    'more than high school':                              'More than high school',
    'less than high school':                              'Less than high school',
    "don't know": np.nan, 'refused': np.nan,
}
def edu_category(v):
    if pd.isna(v): return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return _EDU_NUM.get(int(v), np.nan)
    return _EDU_STR.get(str(v).strip().lower(), np.nan)

if 'Education_level' in df_t1.columns:
    df_t1['Education_Cat'] = df_t1['Education_level'].apply(edu_category)

# Race/Ethnicity: RIDRETH1
# Code 6 (Non-Hispanic Asian) exists only from 2011 onward, so that category
# is structurally absent for earlier cycles — worth noting when interpreting.
_ETH_NUM = {1: 'Mexican American', 2: 'Other Hispanic',
            3: 'Non-Hispanic White', 4: 'Non-Hispanic Black',
            5: 'Other Race', 6: 'Non-Hispanic Asian', 7: 'Other Race'}
def eth_category(v):
    if pd.isna(v): return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return _ETH_NUM.get(int(v), np.nan)
    return str(v).strip()      # already decoded upstream — pass through

if 'Ethnicity' in df_t1.columns:
    df_t1['Ethnicity_Cat'] = df_t1['Ethnicity'].apply(eth_category)

# Print each harmonised column with dropna=False so unmapped values surface
# as NaN counts rather than disappearing silently.
for col in ['Sex_Cat', 'Family_PIR_Cat', 'Education_Cat', 'Ethnicity_Cat']:
    if col in df_t1.columns:
        print(f'\n{col}:\n{df_t1[col].value_counts(dropna=False).to_string()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11b.2 — Comprehensive Table 1 stratified by cluster / subtype
# ══════════════════════════════════════════════════════════════════════
# The publication version: adds an Overall column, explicit missingness rows,
# a named Test column, and fixed category orderings. This is the table intended
# for the manuscript.
from scipy.stats import kruskal, chi2_contingency

# (variable, display label) pairs so units appear in the printed table.
CONT_VARS = [
    ('Age',              'Age (years)'),
    ('BMI',              'BMI (kg/m\u00b2)'),
    ('HbA1c',            'HbA1c (%)'),
    ('Fasting_glucose',  'Fasting glucose (mg/dL)'),
    ('Fasting_insulin',  'Fasting insulin (\u00b5U/mL)'),
    ('HOMA_IR',          'HOMA-IR'),
    ('HOMA_B',           'HOMA-B (%)'),
    ('eGFR',             'eGFR (mL/min/1.73 m\u00b2)'),
    ('Systolic_BP',      'Systolic BP (mmHg)'),
    ('Diastolic_BP',     'Diastolic BP (mmHg)'),
    ('LDL',              'LDL cholesterol (mg/dL)'),
    ('HDL',              'HDL cholesterol (mg/dL)'),
    ('Triglycerides',    'Triglycerides (mg/dL)'),
    ('Diabetes_duration','Diabetes duration (years)'),
]
CAT_VARS = [
    ('Sex_Cat',          'Sex'),
    ('Ethnicity_Cat',    'Race / Ethnicity'),
    ('Education_Cat',    'Education level'),
    ('Family_PIR_Cat',   'Family PIR'),
    ('Smoking_category', 'Smoking status'),
    ('Alcohol_status',   'Alcohol consumption'),
    ('Physical_activity','Physical activity'),
]
# Fixed orderings so ordinal categories read low-to-high rather than
# alphabetically; anything not listed is appended alphabetically below.
CAT_ORDER = {
    'Sex_Cat':        ['Male', 'Female'],
    'Education_Cat':  ['Less than high school', 'High school', 'More than high school'],
    'Family_PIR_Cat': ['Low (PIR < 1.3)', 'Medium (1.3 \u2264 PIR < 3.5)', 'High (PIR \u2265 3.5)'],
    'Ethnicity_Cat':  ['Mexican American', 'Other Hispanic', 'Non-Hispanic White',
                       'Non-Hispanic Black', 'Non-Hispanic Asian', 'Other Race'],
}

clusters  = sorted(df_t1['Consensus_Cluster'].unique())
col_names = [subtype_map.get(c, f'Cluster {c}') for c in clusters]
# Denominators computed once and reused for every percentage below.
cluster_n = {c: int((df_t1['Consensus_Cluster'] == c).sum()) for c in clusters}
total_n   = int(len(df_t1))
rows = []

# Header row — sample sizes
# Conventional Table 1 opener: every percentage below refers to these Ns.
hdr = {'Variable': 'N (total)', 'Level': ''}
for c, cn in zip(clusters, col_names):
    hdr[cn] = str(cluster_n[c])
hdr['Overall'] = str(total_n)
hdr['p-value'] = ''
hdr['Test']    = ''
rows.append(hdr)

# ── Continuous variables ──────────────────────────────────────────────
for var, disp in CONT_VARS:
    if var not in df_t1.columns:
        continue
    r = {'Variable': disp, 'Level': 'median [Q1, Q3]'}
    groups, miss_counts = [], []
    for c, cn in zip(clusters, col_names):
        sub  = df_t1.loc[df_t1['Consensus_Cluster'] == c, var]
        vals = sub.dropna()
        miss_counts.append(int(sub.isna().sum()))   # tracked for the Missing row
        if len(vals) == 0:
            r[cn] = 'N/A'
        else:
            q1, med, q3 = vals.quantile([0.25, 0.50, 0.75])
            r[cn] = f'{med:.1f} [{q1:.1f}, {q3:.1f}]'
        groups.append(vals.values)
    all_vals = df_t1[var].dropna()
    if len(all_vals):
        q1, med, q3 = all_vals.quantile([0.25, 0.50, 0.75])
        r['Overall'] = f'{med:.1f} [{q1:.1f}, {q3:.1f}]'
    else:
        r['Overall'] = 'N/A'
    if all(len(g) >= 2 for g in groups):
        try:
            _, p = kruskal(*groups)
            r['p-value'] = '<0.001' if p < 0.001 else f'{p:.3f}'
            r['Test']    = 'Kruskal-Wallis'   # named explicitly for the methods section
        except Exception:
            r['p-value'] = 'N/A'
            r['Test']    = ''
    else:
        r['p-value'] = 'N/A'
        r['Test']    = ''
    rows.append(r)
    # Missingness row, added only when a variable actually has gaps. Reporting
    # this explicitly is what lets a reader judge whether a median is
    # trustworthy — several NHANES labs are subsample-only.
    if any(n > 0 for n in miss_counts):
        rm = {'Variable': '', 'Level': '    Missing, n (%)'}
        for (c, cn), nm in zip(zip(clusters, col_names), miss_counts):
            rm[cn] = f'{nm} ({100*nm/cluster_n[c]:.1f}%)'
        tm = df_t1[var].isna().sum()
        rm['Overall'] = f'{tm} ({100*tm/total_n:.1f}%)'
        rm['p-value'] = ''
        rm['Test']    = ''
        rows.append(rm)

# ── Categorical variables ─────────────────────────────────────────────
for var, disp in CAT_VARS:
    if var not in df_t1.columns:
        continue
    work = df_t1[['Consensus_Cluster', var]].copy()
    # Make missingness an explicit level so it appears as a table row.
    work[var] = work[var].fillna('Missing')
    explicit = CAT_ORDER.get(var, [])
    observed = list(work[var].unique())
    # Levels not in the explicit ordering are appended alphabetically.
    other    = sorted([lvl for lvl in observed if lvl not in explicit and lvl != 'Missing'])
    levels   = [lvl for lvl in explicit if lvl in observed] + other
    if 'Missing' in observed:
        levels.append('Missing')          # always last
    ct = pd.crosstab(work['Consensus_Cluster'], work[var]).reindex(columns=levels, fill_value=0)
    # Chi-square is computed on NON-MISSING cells only: including a Missing
    # column would test differential missingness rather than the variable.
    ct_nm = ct.drop(columns=['Missing'], errors='ignore')
    try:
        if ct_nm.values.sum() > 0 and ct_nm.shape[1] >= 2:
            _, p_chi, _, _ = chi2_contingency(ct_nm)
            p_str     = '<0.001' if p_chi < 0.001 else f'{p_chi:.3f}'
            test_name = 'Chi-square'
        else:
            p_str, test_name = 'N/A', ''
    except Exception:
        p_str, test_name = 'N/A', ''
    # Header row carries the variable name and the omnibus p; level rows carry
    # the counts.
    hrow = {'Variable': disp, 'Level': 'n (%)', 'p-value': p_str, 'Test': test_name, 'Overall': ''}
    for cn in col_names:
        hrow[cn] = ''
    rows.append(hrow)
    ov = work[var].value_counts()
    for lvl in levels:
        lrow = {'Variable': '', 'Level': f'    {lvl}'}
        for c, cn in zip(clusters, col_names):
            n = int(ct.loc[c, lvl]) if c in ct.index else 0
            lrow[cn] = f'{n} ({100*n/cluster_n[c]:.1f}%)'
        n_t = int(ov.get(lvl, 0))
        lrow['Overall'] = f'{n_t} ({100*n_t/total_n:.1f}%)'
        lrow['p-value'] = ''
        lrow['Test']    = ''
        rows.append(lrow)

table1b = pd.DataFrame(rows)
display_cols = ['Variable', 'Level'] + col_names + ['Overall', 'p-value', 'Test']
table1b = table1b[[c for c in display_cols if c in table1b.columns]]
# Widen the display so long subtype column headers are not truncated.
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 220)
print('=' * 120)
print('TABLE 1 \u2014 BASELINE CHARACTERISTICS STRATIFIED BY CONSENSUS CLUSTER')
print('  Continuous: median [Q1, Q3] + Kruskal-Wallis p')
print('  Categorical: n (%) + Chi-square p  |  Missing reported explicitly')
print('=' * 120)
print(table1b.to_string(index=False))
table1b.to_csv('table1_baseline_characteristics.csv', index=False)
print('\nSaved \u2192 table1_baseline_characteristics.csv')

## 12 · Lipid Profile by Subtype

In [ ]:
# Lipids are NOT clustering inputs, so differences here are external
# validation of the phenotypes rather than a restatement of their definition.
lipid_vars = ['LDL', 'HDL', 'Triglycerides']
lipid_labels = ['LDL (mg/dL)', 'HDL (mg/dL)', 'Triglycerides (mg/dL)']
# Clinical action thresholds per variable, as (value, label, colour).
# HDL is inverted relative to the others: LOW is the adverse direction, and the
# threshold is sex-specific (40 men / 50 women), hence two separate lines.
lipid_thresholds = [
    [(130, 'LDL high (130)', 'orange'), (160, 'LDL very high (160)', 'crimson')],
    [(40,  'HDL low-M (40)', 'crimson'), (50, 'HDL low-F (50)', 'orange')],
    [(150, 'High (150)', 'orange'), (200, 'Very high (200)', 'crimson')],
]

# Copy so the Cluster_Label column does not persist on the main frame.
df_lip = df.copy()
# By now CLUSTER_NAMES is populated, so these axes carry subtype names.
df_lip['Cluster_Label'] = df_lip['Consensus_Cluster'].apply(
    lambda x: cluster_label(x) if CLUSTER_NAMES else str(x)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, var, lbl, thresholds in zip(axes, lipid_vars, lipid_labels, lipid_thresholds):
    if var not in df.columns:
        # Draw a placeholder rather than leaving a blank axis, so the figure
        # still communicates that the variable was unavailable.
        ax.text(0.5, 0.5, f'{var}\nnot available',
                ha='center', va='center', transform=ax.transAxes)
        continue
    violin_box(ax, df_lip, x='Cluster_Label', y=var,
               palette='Set2')
    for thresh, label, color in thresholds:
        ax.axhline(thresh, color=color, lw=1.0, ls='--', alpha=0.75, label=label)
    ax.set_title(f'{lbl} by Subtype', fontweight='bold')
    ax.set_xlabel('Subtype')
    ax.set_ylabel(lbl)
    ax.legend(fontsize=8)
    # Omnibus test appended to the title so significance travels with the figure.
    H, p, _ = kw_dunn(df_lip, var)
    p_str = '<0.001' if p < 0.001 else f'{p:.3f}'
    ax.set_title(ax.get_title() + f'\nKW p{p_str}', fontsize=9)

fig.suptitle('Lipid Profile by T2DM Subtype', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
_save(fig, 'fig09_lipid_profile')
plt.show()

## 13 · Blood Pressure by Subtype

In [ ]:
# Blood pressure is likewise not a clustering input — another external check.
df_bp = df.copy()
df_bp['Cluster_Label'] = df_bp['Consensus_Cluster'].apply(
    lambda x: cluster_label(x) if CLUSTER_NAMES else str(x)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# (axis, column, label, reference line) — 140/90 is the classical hypertension
# threshold; note more recent ACC/AHA guidance uses 130/80, so state which
# definition the manuscript adopts.
for ax, var, lbl, ref in [
    (axes[0], 'Systolic_BP',  'Systolic BP (mmHg)',  140),
    (axes[1], 'Diastolic_BP', 'Diastolic BP (mmHg)',  90),
]:
    if var not in df.columns:
        ax.text(0.5, 0.5, f'{var} not available',
                ha='center', va='center', transform=ax.transAxes)
        continue
    violin_box(ax, df_bp, x='Cluster_Label', y=var, palette='Set1')
    ax.axhline(ref, color='crimson', lw=1.2, ls='--', alpha=0.8,
               label=f'Hypertension threshold ({ref})')
    ax.set_title(f'{lbl} by Subtype', fontweight='bold')
    ax.set_xlabel('Subtype')
    ax.set_ylabel(lbl)
    ax.legend(fontsize=8)
    # dunn is returned but unused here; kw_dunn computes it regardless.
    H, p, dunn = kw_dunn(df_bp, var)
    p_str = '<0.001' if p < 0.001 else f'{p:.3f}'
    ax.set_title(ax.get_title() + f'\nKW p{p_str}', fontsize=9)

fig.suptitle('Blood Pressure by T2DM Subtype', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
_save(fig, 'fig10_blood_pressure')
plt.show()

## 14 · Ethnicity & Lifestyle Factors by Subtype

In [ ]:
from scipy.stats import chi2_contingency

# Sociodemographic and lifestyle composition of each subtype. Differences here
# speak to who the phenotypes describe, which matters for generalisability and
# for the equity framing of the discussion.
cat_plot_vars = [
    ('Ethnicity',         'Ethnicity'),
    ('Smoking_category',  'Smoking Status'),
    ('Alcohol_status',    'Alcohol Consumption'),
    ('Physical_activity', 'Physical Activity'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()   # single index over the 2x2 grid

for ax, (var, lbl) in zip(axes, cat_plot_vars):
    if var not in df.columns:
        ax.text(0.5, 0.5, f'{var}\nnot available',
                ha='center', va='center', transform=ax.transAxes)
        continue

    # Proportional stacked bar per cluster
    # Row-normalised so clusters of very different sizes remain comparable.
    ct = pd.crosstab(df['Consensus_Cluster'], df[var])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

    # Rename index to subtype labels
    ct_pct.index = [cluster_label(c) if CLUSTER_NAMES else str(c)
                    for c in ct_pct.index]

    ct_pct.plot(kind='bar', stacked=True, ax=ax,
                colormap='Set2', edgecolor='white', linewidth=0.5)

    # Chi-square p-value
    # Computed on RAW COUNTS (ct), not percentages — chi-square requires
    # frequencies, and running it on ct_pct would give a meaningless statistic.
    try:
        chi2, p_chi, _, _ = chi2_contingency(ct)
        p_str = '<0.001' if p_chi < 0.001 else f'{p_chi:.3f}'
    except Exception:
        p_str = 'N/A'

    ax.set_title(f'{lbl} by Subtype\nχ² p={p_str}', fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.set_xlabel('Subtype')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
    ax.legend(title=lbl, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

fig.suptitle('Ethnicity & Lifestyle Factors by T2DM Subtype',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
_save(fig, 'fig11_ethnicity_lifestyle')
plt.show()

## 15 · Diabetes Duration by Subtype

In [ ]:
if 'Diabetes_duration' not in df.columns:
    print('Diabetes_duration not available — skipping.')
else:
    # Duration is only defined for participants with a self-reported diagnosis
    # age, so this subset is substantially smaller than the full cohort — an
    # important caveat when reading these panels.
    df_dur = df.dropna(subset=['Diabetes_duration']).copy()
    df_dur['Cluster_Label'] = df_dur['Consensus_Cluster'].apply(
        lambda x: cluster_label(x) if CLUSTER_NAMES else str(x)
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    violin_box(axes[0], df_dur, x='Cluster_Label', y='Diabetes_duration',
               palette='Greens')
    axes[0].set_title('Diabetes Duration by Subtype', fontweight='bold')
    axes[0].set_xlabel('Subtype')
    axes[0].set_ylabel('Diabetes Duration (years)')
    H, p, dunn = kw_dunn(df_dur, 'Diabetes_duration')   # dunn printed at the end
    p_str = '<0.001' if p < 0.001 else f'{p:.3f}'
    axes[0].set_title(axes[0].get_title() + f'\nKW p{p_str}', fontsize=9)

    # Duration categories
    # Bands rather than raw years: duration is heavily right-skewed and
    # rounded to whole years, so categories communicate the shift more clearly.
    def dur_cat(v):
        if pd.isna(v): return 'Unknown'
        if v < 5:   return '< 5 years'
        if v < 10:  return '5 – 9 years'
        if v < 20:  return '10 – 19 years'
        return '≥ 20 years'

    dur_order = ['< 5 years', '5 – 9 years', '10 – 19 years', '≥ 20 years']
    df_dur['Dur_Category'] = df_dur['Diabetes_duration'].apply(dur_cat)
    dur_ct = pd.crosstab(df_dur['Cluster_Label'], df_dur['Dur_Category'])
    dur_pct = dur_ct.div(dur_ct.sum(axis=1), axis=0) * 100
    dur_pct = dur_pct[[c for c in dur_order if c in dur_pct.columns]]
    dur_pct.plot(kind='bar', stacked=True, ax=axes[1],
                 colormap='Greens', edgecolor='white', linewidth=0.5)
    axes[1].set_title('Diabetes Duration Category by Subtype', fontweight='bold')
    axes[1].set_xlabel('Subtype')
    axes[1].set_ylabel('Percentage (%)')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
    axes[1].legend(title='Duration', bbox_to_anchor=(1.02, 1),
                   loc='upper left', fontsize=8)
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))

    fig.suptitle('Diabetes Duration Analysis by Subtype',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    _save(fig, 'fig12_diabetes_duration')
    plt.show()

    # Pairwise results: identifies WHICH subtypes differ in duration, which the
    # omnibus p alone cannot say.
    print('Dunn post-hoc (Diabetes_duration):')
    print(dunn.to_string(index=False))

## 16 · Mortality Analysis by T2DM Subtype

Cause-specific mortality rates per **1,000 person-years** (95 % CI) stratified
by consensus cluster / Ahlqvist subtype, using NHANES linked mortality data.

**Cause definitions**

* **All-cause** — `mortstat == 1`
* **CVD** — `ucod_leading` in {1, 5} (heart disease + cerebrovascular)
* **Diabetes-specific** — `ucod_leading == 7` (diabetes mellitus as leading cause)

**Statistics** — Poisson exact rates; log-rank test between clusters.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 16.1 — Mortality rate table (per 1,000 person-years, Poisson 95 % CI)
#         + multi-group log-rank test
# ══════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from scipy.stats import chi2 as _chi2

# EXTERNAL DEPENDENCY: this file is produced elsewhere (NHANES public-use
# linked mortality file joined to the clustered cohort), not by this notebook.
df1  = pd.read_csv("diabetes_with_mortality.csv")

# eligstat == 1 keeps only participants eligible for mortality follow-up;
# ineligible records have no valid follow-up time and would bias the rates.
df_mort = df1[df1['eligstat'] == 1].copy()
# permth_exm is months from examination to death/censoring; /12 gives the
# person-YEARS that form the denominator of every rate below.
df_mort['py']      = df_mort['permth_exm'] / 12.0
# ucod_leading is the NHANES leading-cause recode: 1 = heart disease,
# 5 = cerebrovascular disease, so {1, 5} together define CVD death.
df_mort['_cvd_ev'] = df_mort['ucod_leading'].isin([1, 5]).astype(float)
df_mort['_dia_ev'] = (df_mort['ucod_leading'] == 7).astype(float)   # 7 = diabetes mellitus

clusters  = sorted(df_mort['Consensus_Cluster'].unique())
col_names = [subtype_map.get(c, f'Cluster {c}') for c in clusters]
cluster_n = {c: int((df_mort['Consensus_Cluster'] == c).sum()) for c in clusters}
total_n   = int(len(df_mort))


def poisson_rate_ci(events, person_years, per=1000, alpha=0.05):
    """Poisson exact rate and 95 % CI per *per* person-years."""
    # Exact (chi-square based) limits rather than a normal approximation:
    # diabetes-specific deaths are few, and the normal approximation is
    # unreliable — and can go negative — at low event counts.
    rate = events / person_years * per
    if events == 0:
        # With zero events the lower limit is 0 and only a one-sided upper
        # limit is defined.
        lo = 0.0
        hi = _chi2.ppf(1 - alpha, 2) / (2 * person_years) * per
    else:
        lo = _chi2.ppf(alpha / 2,     2 * events)       / (2 * person_years) * per
        hi = _chi2.ppf(1 - alpha / 2, 2 * (events + 1)) / (2 * person_years) * per
    return rate, lo, hi


def fmt_rate(events, py):
    # Single formatted cell: rate with its CI, ready for the table.
    r, lo, hi = poisson_rate_ci(events, py)
    return f'{r:.1f} ({lo:.2f}\u2013{hi:.2f})'


def logrank_multigroup(df_sub, time_col, event_col, group_col):
    """
    Overall log-rank test for k groups.
    Returns (chi2_statistic, p_value); degrees of freedom = k - 1.
    """
    # CAUTION: this uses the simplified sum((O-E)^2 / E) statistic, which
    # ignores the covariance between groups. The standard k-sample log-rank
    # uses (O-E)' V^-1 (O-E) with the full variance-covariance matrix, so the
    # p-values here are approximate. See Known issues in the header.
    from scipy.stats import chi2 as chi2s
    groups   = sorted(df_sub[group_col].unique())
    k        = len(groups)
    idx      = {g: df_sub[group_col] == g for g in groups}
    ev_times = sorted(df_sub.loc[df_sub[event_col] == 1, time_col].unique())
    O = np.zeros(k)   # observed events per group
    E = np.zeros(k)   # expected events per group under the null
    for t in ev_times:
        # Risk set at time t: everyone whose follow-up reaches at least t.
        at_risk = np.array([
            int((idx[g] & (df_sub[time_col] >= t)).sum()) for g in groups
        ])
        events = np.array([
            int((idx[g] & (df_sub[time_col] == t)
                        & (df_sub[event_col] == 1)).sum()) for g in groups
        ])
        n = at_risk.sum()
        d = events.sum()
        if n < 2:
            continue          # no comparison possible with fewer than 2 at risk
        # Under the null, deaths distribute across groups in proportion to
        # their share of the risk set.
        E += at_risk * d / n
        O += events
    stat = float(sum((O[j] - E[j]) ** 2 / E[j] for j in range(k) if E[j] > 0))
    return stat, chi2s.sf(stat, df=k - 1)


def p_str(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'


# ── Cause definitions ─────────────────────────────────────────────────
# (label, boolean mask over df_mort, event column name for the log-rank)
CAUSES = [
    ('All-cause mortality',         df_mort['mortstat'] == 1,  'mortstat'),
    ('CVD mortality',               df_mort['_cvd_ev'] == 1,   '_cvd_ev'),
    ('Diabetes-specific mortality', df_mort['_dia_ev'] == 1,   '_dia_ev'),
]

rows = []
for cause_label, overall_mask, ev_col in CAUSES:
    _, p_lr = logrank_multigroup(df_mort, 'py', ev_col, 'Consensus_Cluster')
    ev_all  = int(overall_mask.sum())
    py_all  = df_mort['py'].sum()

    row = {
        'Mortality outcome': cause_label,
        'Overall':           fmt_rate(ev_all, py_all),
        'p-value':           p_str(p_lr),
    }
    for c, cn in zip(clusters, col_names):
        sub     = df_mort[df_mort['Consensus_Cluster'] == c]
        # Index the mask by the subset's index to get this cluster's events.
        ev      = int(overall_mask[sub.index].sum())
        row[cn] = fmt_rate(ev, sub['py'].sum())
    rows.append(row)

    # Event count sub-row
    # Raw counts matter: a rate built on very few events is unstable however
    # tight the point estimate looks.
    n_row = {'Mortality outcome': '    Events, n', 'Overall': str(ev_all), 'p-value': ''}
    for c, cn in zip(clusters, col_names):
        sub       = df_mort[df_mort['Consensus_Cluster'] == c]
        n_row[cn] = str(int(overall_mask[sub.index].sum()))
    rows.append(n_row)

mort_table = pd.DataFrame(rows)
display_cols = ['Mortality outcome'] + col_names + ['Overall', 'p-value']
mort_table   = mort_table[[c for c in display_cols if c in mort_table.columns]]

# ── Print person-years summary ────────────────────────────────────────
# Exposure per cluster: the denominator behind every rate, and the quantity a
# reviewer will want reported alongside them.
print('Person-years at risk:')
for c, cn in zip(clusters, col_names):
    py = df_mort[df_mort['Consensus_Cluster'] == c]['py'].sum()
    print(f'  {cn:<44}: {py:>9,.0f} py   (n = {cluster_n[c]})')
print(f'  {"Overall":<44}: {df_mort["py"].sum():>9,.0f} py   (n = {total_n})')
print()

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 240)
print('=' * 130)
print('TABLE 2 \u2014 MORTALITY RATES PER 1,000 PERSON-YEARS (95 % CI) BY T2DM SUBTYPE')
print('  95 % CI: Poisson exact method.   p-value: log-rank test (chi-square, df = k \u2212 1).')
print('=' * 130)
print(mort_table.to_string(index=False))

mort_table.to_csv('table2_mortality_rates.csv', index=False)
print('\nSaved \u2192 table2_mortality_rates.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 16.2 — Kaplan–Meier survival curves (all-cause, CVD, diabetes)
# ══════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt


def kaplan_meier(times, events):
    """
    Compute KM estimator.
    Returns (time_array, survival_probability_array).
    """
    # Product-limit estimator: survival is multiplied down by (1 - 1/n_risk)
    # at each observed death, with censored observations leaving the risk set
    # silently. Applying single decrements successively over tied deaths is
    # algebraically equal to the standard (1 - d/n) update.
    # LIMITATION: returns point estimates only — no confidence bands and no
    # at-risk table, both of which are expected in a published KM figure.
    order  = np.argsort(times)
    t_s    = np.array(times)[order]
    e_s    = np.array(events)[order]
    n      = len(t_s)
    t_out  = [0.0]     # curves start at S(0) = 1
    s_out  = [1.0]
    S      = 1.0
    for i, (t, e) in enumerate(zip(t_s, e_s)):
        if e == 1:
            # Everyone earlier in the sorted order has already left the risk
            # set, whether by death or censoring.
            n_risk = n - i
            S *= (1.0 - 1.0 / n_risk)
            t_out.append(t)
            s_out.append(S)
    return np.array(t_out), np.array(s_out)


# One colour per cluster, consistent across all three panels.
PALETTE = plt.colormaps['tab10'].resampled(max(len(clusters) + 1, 4))
COLORS  = [PALETTE(i) for i in range(len(clusters))]

CAUSE_DEFS = [
    ('All-cause mortality',         'mortstat'),
    ('CVD mortality',               '_cvd_ev'),
    ('Diabetes-specific mortality', '_dia_ev'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (title, ev_col) in zip(axes, CAUSE_DEFS):
    for c, cn, col in zip(clusters, col_names, COLORS):
        sub   = df_mort[df_mort['Consensus_Cluster'] == c].dropna(subset=['py', ev_col])
        t_km, s_km = kaplan_meier(sub['py'].values, sub[ev_col].values.astype(int))
        label = cn.split(' (')[0]     # strip the parenthetical to keep the legend short
        # where='post': survival is constant between events and drops AT the
        # event time, which is the correct step convention for KM.
        ax.step(t_km, s_km, where='post', color=col, linewidth=1.8, label=label)

    ax.set_xlabel('Follow-up (years)', fontsize=10)
    ax.set_ylabel('Survival probability', fontsize=10)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(left=0)
    # Median-survival reference. In this cohort the curves stay well above 0.5
    # for the cause-specific outcomes, so no median is reached.
    ax.axhline(0.5, color='grey', ls=':', lw=0.8, alpha=0.6, label='S = 0.50')
    ax.legend(fontsize=8, framealpha=0.9)

    # Log-rank p annotation
    _, p_lr   = logrank_multigroup(df_mort, 'py', ev_col, 'Consensus_Cluster')
    p_label   = 'p < 0.001' if p_lr < 0.001 else ('p = ' + f'{p_lr:.3f}')
    ann_text  = 'Log-rank' + chr(10) + p_label   # chr(10) = newline
    ax.text(0.97, 0.97, ann_text,
            transform=ax.transAxes, ha='right', va='top', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      alpha=0.85, edgecolor='0.7'))

fig.suptitle('Kaplan\u2013Meier Survival Curves by T2DM Subtype',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
_save(fig, 'fig13_kaplan_meier')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 16.3 — Mortality rate dot-and-CI plot, grouped by cause of death
# ══════════════════════════════════════════════════════════════════════
# Complements the KM curves: those show survival over time, this shows the
# summary rate with its uncertainty, which is what supports a claim that one
# subtype's mortality differs from another's.
import numpy as np
import matplotlib.pyplot as plt

CAUSE_DEFS = [
    ('All-cause mortality',         df_mort['mortstat'] == 1),
    ('CVD mortality',               df_mort['_cvd_ev'] == 1),
    ('Diabetes-specific mortality', df_mort['_dia_ev'] == 1),
]

PALETTE = plt.colormaps['tab10'].resampled(max(len(clusters) + 1, 4))
COLORS  = [PALETTE(i) for i in range(len(clusters))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, mask) in zip(axes, CAUSE_DEFS):
    for y_cur, (c, cn, col) in enumerate(zip(clusters, col_names, COLORS)):
        sub       = df_mort[df_mort['Consensus_Cluster'] == c]
        ev        = int(mask[sub.index].sum())
        r, lo, hi = poisson_rate_ci(ev, sub['py'].sum())
        # xerr as distances from the point, since the exact Poisson interval is
        # asymmetric and cannot be expressed as a single +/- value.
        ax.errorbar(r, y_cur,
                    xerr=[[r - lo], [hi - r]],
                    fmt='o', color=col, markersize=7,
                    elinewidth=1.5, capsize=4, capthick=1.5)
        # Numeric label placed past the upper limit so it never sits on the bar.
        ax.text(hi + 0.15, y_cur, f'{r:.1f}',
                va='center', fontsize=8.5, color=col)

    # Overall reference line
    # Cohort-wide rate: shows at a glance which subtypes sit above or below average.
    r_all = int(mask.sum()) / df_mort['py'].sum() * 1000
    ax.axvline(r_all, color='grey', ls='--', lw=1.0, alpha=0.7,
               label='Overall (' + f'{r_all:.1f}' + ')')

    ax.set_yticks(range(len(clusters)))
    ax.set_yticklabels([cn.split(' (')[0] for cn in col_names], fontsize=9)
    ax.set_xlabel('Rate per 1,000 person-years', fontsize=10)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, framealpha=0.9)
    ax.invert_yaxis()   # first cluster at the top, matching the table order

fig.suptitle('Mortality Rates per 1,000 Person-Years by T2DM Subtype',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
_save(fig, 'fig14_mortality_rates')
plt.show()

In [ ]:
# ── Export labelled dataset ───────────────────────────────────────────────────
# Carries the Ahlqvist subtype labels plus every derived clinical category
# created above, so downstream notebooks need not recompute them.
out_csv = "t2d_ahlqvist_mapped.csv"
df.to_csv(out_csv, index=False)

# Verify WTMEC2YR passes through
# The survey weight is not used for any analysis in THIS notebook, but must
# survive to the export for the weighted sensitivity analysis downstream.
if 'WTMEC2YR' in df.columns:
    print(f'✅ WTMEC2YR present in {out_csv}')
else:
    print('⚠️  WTMEC2YR not in export — run upstream pipeline first')
print(f"Exported : {out_csv}  ({df.shape[0]} rows × {df.shape[1]} cols)")

# Summary of all saved figures
# Inventory as a final check that every figure was written and none failed silently.
print(f"\nFigures in '{FIGURE_DIR}/':")
for fname in sorted(os.listdir(FIGURE_DIR)):
    kb = os.path.getsize(os.path.join(FIGURE_DIR, fname)) / 1024
    print(f"  {fname:<55}  {kb:6.1f} KB")